In [5]:
"""
This notebook creates the dimension tables of our data model. This tables are not expected to change, so we only create them once using this notebook
and in theory it should never be executed again unless the ISTAC changes something like a code or the name of some airport
"""

'\nThis notebook creates the dimension tables of our data model. This tables are not expected to change, so we only create them once using this notebook\nand in theory it should never be executed again unless the ISTAC changes something like a code or the name of some airport\n'

In [6]:
import pandas as pd
import utils as u

# Airport

In [7]:
url_passengers = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000001/~latest.csv?lang=en"
url_gm = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000002/~latest.csv"
url_op = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000003/~latest.csv"

In [8]:
data_t = pd.read_csv(u.get_data_from_API_call(url_passengers))

/tmp/ipykernel_1267/3482215424.py:1: DtypeWarning: Columns (19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  data_t = pd.read_csv(u.get_data_from_API_call(url_passengers))


In [9]:
df = data_t.copy(deep=True)

In [10]:
df.drop(columns=df.columns[df.columns.str.endswith('#es')], inplace=True)

In [11]:
df['AEROPUERTO_ESCALA_CODE']

0          FOREIGN
1          FOREIGN
2          FOREIGN
3          FOREIGN
4          FOREIGN
            ...   
2984683         RU
2984684         RU
2984685         RU
2984686         RU
2984687         RU
Name: AEROPUERTO_ESCALA_CODE, Length: 2984688, dtype: object

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2984688 entries, 0 to 2984687
Data columns (total 16 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   SERVICIO_AEREO#en                object 
 1   SERVICIO_AEREO_CODE              object 
 2   MEDIDAS#en                       object 
 3   MEDIDAS_CODE                     object 
 4   MOVIMIENTO_AERONAVE#en           object 
 5   MOVIMIENTO_AERONAVE_CODE         object 
 6   TIME_PERIOD#en                   object 
 7   TIME_PERIOD_CODE                 object 
 8   AEROPUERTO_BASE#en               object 
 9   AEROPUERTO_BASE_CODE             object 
 10  AEROPUERTO_ESCALA#en             object 
 11  AEROPUERTO_ESCALA_CODE           object 
 12  OBS_VALUE                        float64
 13  ESTADO_OBSERVACION#en            object 
 14  ESTADO_OBSERVACION_CODE          object 
 15  CONFIDENCIALIDAD_OBSERVACION#en  float64
dtypes: float64(2), object(14)
memory usage: 364.3+ MB


Steps to getting the airports right:
1) Delete countries, autonomous communities and "Rest of"/"Remain of"
2) Join with airports from ourairports using word similitude or something like that

[Understanding _CODE from airports](https://www3.gobiernodecanarias.org/aplicaciones/appsistac/activos-semanticos/codelists/codelists/ISTAC/CL_AEROPUERTOS/01.004/detail)

I think I can do all I did (selecting only airports) with the API, with the "granularity" option, tomorrow ill check

Hello, it is tomorrow, it can not be done, I tried

In [13]:
# Extract countries
countries = df.loc[df['AEROPUERTO_ESCALA_CODE'].str.match(r'^[A-Z]{2}$', na=False), ['AEROPUERTO_ESCALA#en', 'AEROPUERTO_ESCALA_CODE']]
countries.drop_duplicates(inplace=True)
countries.rename({'AEROPUERTO_ESCALA#en': 'CountryName', 'AEROPUERTO_ESCALA_CODE': 'iso_country'}, inplace=True, axis=1)

In [14]:
countries

,CountryName,iso_country
2256,Gambia,GM
3384,Netherlands,NL
6486,Belgium,BE
8460,Morocco,MA
10716,Norway,NO
11562,Ukraine,UA
16074,Lithuania,LT
16638,Slovakia,SK
18330,Germany,DE
23124,Austria,AT


In [15]:
# Rest of/Remain of AEROPUERTO_ESCALA#en have "_O" at the end of their code
df_f = df.loc[~df['AEROPUERTO_ESCALA_CODE'].str.endswith('_O')]

# Delete entries with the whole country
df_f = df_f.loc[~df_f['AEROPUERTO_ESCALA_CODE'].str.match(r'^[A-Z]{2}$', na=False)]

# Delete autonomous communities (Their code is like ES[0-9][0-9]) 
df_f = df_f.loc[~df_f['AEROPUERTO_ESCALA_CODE'].str.match(r'^ES[0-9]{2}$', na=False)]

# Delete sum of entire island
df_f = df_f.loc[~df_f['AEROPUERTO_ESCALA_CODE'].str.match(r'^ES70[0-9]$', na=False)]

# Delete sum of all autonomous communities and sum of all islands
df_f = df_f.loc[~((df_f['AEROPUERTO_ESCALA_CODE'] == 'ES_XES70') | (df_f['AEROPUERTO_ESCALA_CODE'] == 'ES70') | (df_f['AEROPUERTO_ESCALA_CODE'] == 'FOREIGN'))]

In [16]:
df_f.loc[~df_f['AEROPUERTO_ESCALA#en'].str.endswith('Airport', na=False), 'AEROPUERTO_ESCALA#en'].unique()

array(['Zaragoza Air Base', 'Sandefjord Airport, Torp',
       'Harstad/Narvik Airport, Evenes', 'Stavanger Airport Sola',
       'Amsterdam Airport Schiphol', 'Václav Havel Airport Prague',
       'Moss Airport, Rygge', 'Bergen Airport Flesland',
       'Trondheim Airport Vèrnes'], dtype=object)

In [17]:
istac_airports = df_f[['AEROPUERTO_ESCALA#en', 'AEROPUERTO_ESCALA_CODE']].copy(deep=True)

In [18]:
istac_airports.drop_duplicates(inplace=True)

In [19]:
istac_airports

,AEROPUERTO_ESCALA#en,AEROPUERTO_ESCALA_CODE
282,Maastricht Aachen Airport,NL_EHBK
564,Stuttgart Airport,DE_EDDS
846,Verona Villafranca Airport,IT_LIPX
1128,Metz-Nancy-Lorraine Airport,FR_LFJL
1410,Linz Hörsching Airport,AT_LOWL
...,...,...
79806,Münster Osnabrück Airport,DE_EDDG
80934,Alicante International Airport,ES_LEAL
81498,Paderborn Lippstadt Airport,DE_EDLP
82062,Hamburg Airport,DE_EDDH


In [20]:
istac_airports['ident'] = istac_airports['AEROPUERTO_ESCALA_CODE'].str[3:]

In [21]:
airport_csv = pd.read_csv('airports.csv')

In [22]:
airport_csv = airport_csv.merge(countries, on='iso_country')

In [23]:
join = istac_airports.merge(airport_csv, on='ident', suffixes=("l", "r"))

In [24]:
join

,AEROPUERTO_ESCALA#en,AEROPUERTO_ESCALA_CODE,ident,id,type,name,latitude_deg,longitude_deg,elevation_ft,continent,...,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords,CountryName
0,Maastricht Aachen Airport,NL_EHBK,EHBK,2515,medium_airport,Maastricht Aachen Airport,50.911701,5.770140,375.0,EU,...,Maastricht,yes,EHBK,MST,EHBK,NaN,NaN,https://en.wikipedia.org/wiki/Maastricht_Aache...,NaN,Netherlands
1,Stuttgart Airport,DE_EDDS,EDDS,2222,large_airport,Stuttgart Airport,48.689899,9.221960,1276.0,EU,...,Stuttgart,yes,EDDS,STR,EDDS,NaN,http://www.flughafen-stuttgart.de/,https://en.wikipedia.org/wiki/Stuttgart_Airport,NaN,Germany
2,Verona Villafranca Airport,IT_LIPX,LIPX,4366,large_airport,Verona Villafranca Valerio Catullo Airport,45.394955,10.887303,239.0,EU,...,Caselle (VR),yes,LIPX,VRN,LIPX,VR10,http://www.aeroportoverona.it/,https://en.wikipedia.org/wiki/Verona_Airport,"Valerio Catullo, Villafranca International Air...",Italy
3,Metz-Nancy-Lorraine Airport,FR_LFJL,LFJL,4120,medium_airport,Metz-Nancy-Lorraine Airport,48.982101,6.251320,870.0,EU,...,Goin,yes,LFJL,ETZ,LFJL,NaN,https://lorraineaeroport.com/,https://en.wikipedia.org/wiki/Metz-Nancy-Lorra...,NaN,France
4,Linz Hörsching Airport,AT_LOWL,LOWL,4432,medium_airport,Linz-Hörsching Airport / Vogler Air Base,48.233200,14.187500,980.0,EU,...,Linz,yes,LOWL,LNZ,LOWL,NaN,http://www.flughafen-linz.at/,https://en.wikipedia.org/wiki/Linz_Airport,"Blue Danube Airport, Vogler Air Base, Fliegerh...",Austria
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,Münster Osnabrück Airport,DE_EDDG,EDDG,2213,medium_airport,Münster Osnabrück Airport,52.134602,7.684830,160.0,EU,...,Greven,yes,EDDG,FMO,EDDG,NaN,https://www.fmo.de/,https://en.wikipedia.org/wiki/M%C3%BCnster_Osn...,NaN,Germany
190,Alicante International Airport,ES_LEAL,LEAL,3997,large_airport,Alicante-Elche Miguel Hernández Airport,38.282200,-0.558156,142.0,EU,...,Alicante,yes,LEAL,ALC,LEAL,NaN,https://www.aena.es/en/alicante-elche.html,https://en.wikipedia.org/wiki/Alicante–Elche_M...,NaN,Spain
191,Paderborn Lippstadt Airport,DE_EDLP,EDLP,2244,medium_airport,Paderborn Lippstadt Airport,51.614101,8.616320,699.0,EU,...,Büren,yes,EDLP,PAD,EDLP,NaN,NaN,https://en.wikipedia.org/wiki/Paderborn_Lippst...,NaN,Germany
192,Hamburg Airport,DE_EDDH,EDDH,2214,large_airport,Hamburg Helmut Schmidt Airport,53.630402,9.988230,53.0,EU,...,Hamburg,yes,EDDH,HAM,EDDH,NaN,https://www.hamburg-airport.de/en/,https://en.wikipedia.org/wiki/Hamburg_Airport,Hamburg-Fuhlsbüttel Airport,Germany


Only two missing airports, I can input them by hand

In [25]:
print(f"Mising airports: {len(istac_airports['AEROPUERTO_ESCALA#en'].unique()) - len(join['AEROPUERTO_ESCALA#en'].unique())}")
missing_airports = set(istac_airports['AEROPUERTO_ESCALA#en'].unique()) - set(join['AEROPUERTO_ESCALA#en'].unique())
print(f"Missing airports: {missing_airports}")

Mising airports: 4
Missing airports: {'Hassan I Airport', 'Dakhla Airport', 'Berlin-Tegel Airport', 'Robin Hood Doncaster Sheffield Airport'}


In [26]:
result_df = join[['AEROPUERTO_ESCALA#en', 'AEROPUERTO_ESCALA_CODE', 'latitude_deg', 'longitude_deg','iso_country', 'CountryName']].copy(deep=True)
result_df.rename({'iso_country': 'CountryCode', 'AEROPUERTO_ESCALA#en': 'AirportName', 
                  'latitude_deg': 'Latitude', 'longitude_deg': 'Longitude', 
                  'country': 'CountryName', 'AEROPUERTO_ESCALA_CODE': 'AirportCode'}, axis=1, inplace=True)

In [27]:
result_df['AirportId'] = result_df.index

In [28]:
result_df.to_csv('../../data/Airport.csv', index=False)
result_df

,AirportName,AirportCode,Latitude,Longitude,CountryCode,CountryName,AirportId
0,Maastricht Aachen Airport,NL_EHBK,50.911701,5.770140,NL,Netherlands,0
1,Stuttgart Airport,DE_EDDS,48.689899,9.221960,DE,Germany,1
2,Verona Villafranca Airport,IT_LIPX,45.394955,10.887303,IT,Italy,2
3,Metz-Nancy-Lorraine Airport,FR_LFJL,48.982101,6.251320,FR,France,3
4,Linz Hörsching Airport,AT_LOWL,48.233200,14.187500,AT,Austria,4
...,...,...,...,...,...,...,...
189,Münster Osnabrück Airport,DE_EDDG,52.134602,7.684830,DE,Germany,189
190,Alicante International Airport,ES_LEAL,38.282200,-0.558156,ES,Spain,190
191,Paderborn Lippstadt Airport,DE_EDLP,51.614101,8.616320,DE,Germany,191
192,Hamburg Airport,DE_EDDH,53.630402,9.988230,DE,Germany,192


# Territory, AircraftMovement, AirService

In [29]:
url = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000013/~latest.csv?lang=en"

In [30]:
data = pd.read_csv(u.get_data_from_API_call(url))

In [31]:
df = data.copy(deep=True)

In [32]:
df_terr = pd.concat([df[['TERRITORIO_CODE', 'TERRITORIO#en']], df[['AEROPUERTO_ESCALA_CODE', 'AEROPUERTO_ESCALA#en']].rename({'AEROPUERTO_ESCALA#en': 'TERRITORIO#en', 'AEROPUERTO_ESCALA_CODE': 'TERRITORIO_CODE'}, axis=1)]).drop_duplicates()

df_terr.reset_index(inplace=True, drop=True)

# Remove "Total", "Foreign and Spain (Canary Islands excluded)", "Spain"
df_terr = df_terr.loc[~df_terr['TERRITORIO_CODE'].isin(["_T_XES70", "ES", "_T"])]

df_terr['TerritoryId'] = df_terr.index

df_terr.rename({'TERRITORIO_CODE': 'TerritoryCode', 'TERRITORIO#en': 'TerritoryName'}, inplace=True, axis=1)

df_terr = df_terr[['TerritoryId', 'TerritoryCode', 'TerritoryName']]

df_terr.to_csv('../../data/Final_Territory.csv', index=False)

In [33]:
df_am = df[['MOVIMIENTO_AERONAVE_CODE', 'MOVIMIENTO_AERONAVE#en']].drop_duplicates().reset_index(drop=True)

df_am['AircraftMovementId'] = df_am.index

df_am.rename({'MOVIMIENTO_AERONAVE_CODE': 'AircraftMovementCode', 'MOVIMIENTO_AERONAVE#en': 'AircraftMovement'}, axis=1, inplace=True)

df_am = df_am[['AircraftMovementId', 'AircraftMovementCode', 'AircraftMovement']]

df_am.to_csv('../../data/Final_AircraftMovement.csv', index=False)

In [34]:
df_as = df[['SERVICIO_AEREO_CODE', 'SERVICIO_AEREO#en']].drop_duplicates().reset_index(drop=True)

df_as['AirServiceId'] = df_as.index

df_as.rename({'SERVICIO_AEREO_CODE': 'AirServiceCode', 'SERVICIO_AEREO#en': 'AirService'}, axis=1, inplace=True)

df_as = df_as[['AirServiceId', 'AirServiceCode','AirService', ]]

df_as.to_csv('../../data/AirService.csv', index=False)